# Homework 2: Vector Search

## Setup

In [ ]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

## Q1. Embedding a query

The embedder returns a vector of 384 numbers. What's the first value `v[0]`?
* -0.31
* **-0.02**
* 0.12
* 0.44

In [ ]:
from embedder import Embedder

query = "How does approximate nearest neighbor search work?"

embedder = Embedder()
v_query = embedder.encode(query)
v_query[0]

## Q2. Cosine similarity

Take the page `02-vector-search/lessons/07-sqlitesearch-vector.md`, embed its `content`, and compute the cosine similarity with the query vector from Q1. What do you get?
* 0.07
* **0.37**
* 0.68
* 0.92

In [ ]:
target_filename = "02-vector-search/lessons/07-sqlitesearch-vector.md"

page = next(
    doc for doc in documents
    if doc['filename'] == target_filename
)

content = page['content']
v_doc = embedder.encode(content)

v_query.dot(v_doc)

## Q3. Chunking and search by hand

Which file does the highest-scoring chunk belong to (its `filename`)?
* 02-vector-search/lessons/03-embeddings-dataset.md
* 02-vector-search/lessons/06-rag-vector.md
* **02-vector-search/lessons/07-sqlitesearch-vector.md**
* 02-vector-search/lessons/09-onnx-embedder.md

In [ ]:
from gitsource import chunk_documents
import numpy as np

chunks = chunk_documents(documents, size=2000, step=1000)
texts = [chunk['content'] for chunk in chunks]
v_chunks = embedder.encode_batch(texts)

In [ ]:
scores = v_chunks.dot(v_query)
idx = np.argmax(scores)
idx, scores[idx], chunks[idx]

## Q4. Vector search with minsearch

Which file is the `filename` of the first result?
* 02-vector-search/lessons/04-vector-search.md
* **04-evaluation/lessons/05-search-metrics.md**
* 04-evaluation/lessons/13-llm-as-judge.md
* 05-monitoring/lessons/04-metrics.md

In [ ]:
from minsearch import Index, VectorSearch

v_index = VectorSearch(keyword_fields=['content'])
v_index.fit(v_chunks, chunks)

q = "What metric do we use to evaluate a search engine?"
v_q = embedder.encode(q)

results = v_index.search(v_q)
results[0]

## Q5. Text search vs vector search

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?
* 02-vector-search/lessons/01-intro.md
* 02-vector-search/lessons/02-embeddings.md
* **02-vector-search/lessons/08-pgvector.md**
* 03-orchestration/lessons/05-rag.md

In [ ]:
index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(chunks)

q = "How do I store vectors in PostgreSQL?"
v_q = embedder.encode(q)

text_results = index.search(q, num_results=5)
vector_results = v_index.search(v_q, num_results=5)

In [ ]:
text_filenames = {result['filename'] for result in text_results}
vector_filenames = {result['filename'] for result in vector_results}

vector_only_filenames = vector_filenames - text_filenames
vector_only_filenames

## Q6. Hybrid search

Which file is ranked first after Reciprocal Rank Fusion (RRF)?
* 01-agentic-rag/lessons/01-intro.md
* **01-agentic-rag/lessons/13-function-calling.md**
* 01-agentic-rag/lessons/14-agentic-loop.md
* 01-agentic-rag/lessons/16-other-frameworks.md

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
q = "How do I give the model access to tools?"
v_q = embedder.encode(q)

text_results = index.search(q, num_results=5)
vector_results = v_index.search(v_q, num_results=5)

results = rrf([vector_results, text_results])
results[0]